# ② 예보 피처 선택 기록 (E2)

담당: 세민 · 실험 기간: 2026-10-30 ~ 11-01 · 비교 기준: 기준 모델 `E1-01x2`

**목적**: 기상 예보에서 "그 시각에 구름이 얼마나 낄지"를 더 잘 알려주는 입력값을 찾아, 가상 평가연도(2024-06 ~ 2025-05) 벌금을 줄인다.

**결론 먼저**: 하루 요약 피처 11개(`add_weather_features`, 실험 E2-40)를 채택했다. 그날의 예보 요약 5개, **전날 예보 요약 4개**, **아침 안개 신호 2개**다.
- 가상 평가연도: 시드 4개 평균으로 기준 모델보다 벌금 **−6.83%**, 11~15시 −6.83%
- 다른 해(2023-06 ~ 2024-05) 백테스트: 시드 0·1 **−6.41%**, 시드 2·3 −6.38%
- 측정 4개(두 해 × 시드 두 쌍) 평균 **−6.61%**. 직전 최종안 E2-24(−6.19%)를 네 측정 모두에서 이겼다.
- **후보 선택에 한 번도 쓰지 않은 2022-23년(마지막 시험)에서도 비교한 후보 중 1위**다(−3.33%, 시드 두 쌍 모두 1위). 결과를 보기 전에 판단 규칙을 정해 두고 확인했다.
- 안개 신호를 쪼개도, 다른 값을 더해도 나빠졌다(11·12라운드). 단순화(E2-46)는 마지막 시험에서 덜 좋아 채택하지 않았다.
- 개선 폭은 해마다 다르다(2022-23년 −3.3%, 다른 두 해 −6.4 ~ −7.0%). 실제 평가 기간에서는 **3 ~ 7% 사이**로 보는 것이 정직하다.

팀 문서의 6개 아이디어에서 출발해 피처 조합 49개를 시험했다. 여기에 시드를 바꾼 재실행, 세 해에 걸친 백테스트, 안개 시간대 민감도 점검 2개, 최종 확인 2번을 더했다.

| 라운드 | 질문 | 결과 |
|---|---|---|
| 1 | 팀 문서의 6개 아이디어 중 효과 있는 것은? | 하루 요약(E2-04)만 기준 충족 (−2.05%) |
| 2 | 하루 요약의 어느 요소가 효과를 내나? 더 넓히면? | 세 요소를 함께 쓸 때만 크고, 다른 하루 변수를 얹으면 나빠짐 |
| 3 | 강수 합이 필요한가? 다른 피처를 쌓으면? | 강수 합 필요, 쌓기는 효과 없음 |
| 4 | 하루 요약을 다르게 계산하면? | 두 제공사 값을 쓰면 더 좋아짐 (E2-15, −3.10%) |
| 5 | 두 제공사 버전에서 뺄 것·더할 것은? | B사 하루 운량을 빼도 같음 → 중간 결론 E2-16 (−3.44%) |
| 6 | E2-16 위에 더할 하루 정보는? | **전날 강수 합**이 크게 좋아짐 (E2-21, −5.43%) |
| 7 | 전날 정보를 넓히면? | 전날 운량·일교차까지 → **E2-24 (−6.74%)** |
| 8 | 실적이 빠진 행 때문에 생긴 착시는 아닌가? | 관측지점 예보로 다시 계산해도 같은 수준 → 착시 아님 |
| 백테스트 | 다른 해에서도 순위가 유지되나? | 유지 (E2-24 −6.35%) |
| 9 | E2-24 위에 새 정보(변화량·습도·기압·풍속)를 더하면? | 다섯 개 모두 나빠짐 (+0.4 ~ +1.7%) |
| 10 | E2-24에서 덜어내도 되는 것은? | 어느 묶음을 빼도 나빠짐 (+0.5 ~ +2.7%) |
| 중간 검증 | 가장 가까운 두 후보와 측정 4개로 비교하면? | E2-24 평균 −6.19%로 유지 (차이 0.1%p 안) |
| 11 | 아직 안 쓴 정보(강설, 밤사이 비, 아침 안개, 주변 발전소)는? 시각별 풍속·기압을 빼면? | **아침 안개 신호(E2-40)** 만 좋아짐 |
| E2-40 검증 | 측정 4개로도 E2-24보다 좋은가? | 네 측정 모두 좋음 (평균 −6.61%) → **최종** |
| 12 | 안개 신호를 쪼개거나 다듬으면? | 시정·습도 둘 다 있어야 하고, 운고를 더하면 나빠짐 |
| 단순화 | E2-40에서 전날 일교차를 빼도 같은가? (E2-46) | 두 해는 같지만 마지막 시험에서 덜 좋음 → 채택 안 함 |
| 마지막 시험 | 고르는 데 한 번도 안 쓴 2022-23년에서도 순서가 유지되나? | 유지. E2-40이 시드 두 쌍 모두 1위 (−3.33%) |
| 민감도 | 안개 시간대(06~10시)가 우연히 맞은 것은 아닌가? | 이른 아침 창(05~09시)도 좋음. 늦은 창(07~11시)은 약함 → 물리적으로 맞음 |

## 0. 판단 기준

- 모든 실험은 기준 모델(`--base champion` = `E1-01x2`: 두 발행분 예보를 한 모델에, L1 손실) 위에 **변경 하나**만 얹었다.
- 각 실험은 **시드 2개**(0·1) 예측의 평균으로 채점했다. 채점은 팀 채점기 `evaluate.py` 그대로다(참조 구성, 벌금 = 0.10 × Σ SMP·k·|예측 − 실적|).
- 팀 채택 규칙: **벌금이 기준보다 0.5% 이상 낮고, 11~15시 벌금도 낮을 것.**
- 0.5%를 쓰는 이유: 설정은 그대로 두고 시드만 바꿔도 벌금이 이 정도 움직인다(바로 아래 셀에서 확인).
- 아래 표의 판정 열: `채택 기준 충족` / `잡음 범위`(−0.5% < 변화 < +0.5%) / `악화`(+0.5% 이상).
- 판정은 **그 표의 비교 기준에 대한 것**이다. 예를 들어 E2-04를 넓힌 실험은 기준 모델보다 좋아서 "충족"으로 나오지만, E2-04보다 나쁘면 채택하지 않는다.
- `나아진 달`: 12개월 중 비교 기준보다 벌금이 낮았던 달의 수.
- 최종 후보끼리는 시드 4개 평균과 다른 해 백테스트까지 보고 고른다(가상 평가연도 하나로만 고르면 그해에만 맞는 것을 고를 수 있다).

In [1]:
import sys, inspect
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent            # 이 노트북은 analysis_lab/ 에서 실행한다
sys.path.insert(0, str(ROOT / "src_tools"))
from evaluate import load_actual, read_preds, evaluate_preds
import features_weather as fw

PREDS = ROOT / "outputs" / "preds"
ACTUAL = load_actual()
pd.set_option("display.max_colwidth", 70)
_cache = {}

def score(exp):
    """outputs/preds/{exp}.parquet 을 팀 채점기로 채점 (참조 구성)."""
    if exp not in _cache:
        _cache[exp] = evaluate_preds(read_preds(PREDS / f"{exp}.parquet"), actual=ACTUAL)
    return _cache[exp]

def pct(a, b):
    return (a / b - 1) * 100

def judge(d, dpeak):
    if d <= -0.5 and dpeak < 0:
        return "채택 기준 충족"
    return "악화" if d >= 0.5 else "잡음 범위"

def table(rows, ref="E1-01x2"):
    b = score(ref)
    out = []
    for exp, desc in rows:
        r = score(exp)
        d, dp = pct(r["penalty"], b["penalty"]), pct(r["peak"], b["peak"])
        out.append({"실험": exp, "바꾼 것": desc, "벌금(원)": f"{r['penalty']:,.0f}",
                    f"{ref} 대비": f"{d:+.2f}%", "11~15시 대비": f"{dp:+.2f}%",
                    "나아진 달": f"{int((r['by_month'] < b['by_month']).sum())}/12", "판정": judge(d, dp)})
    return pd.DataFrame(out).set_index("실험")

b = score("E1-01x2")
s = score("E1-01s2x2")
print(f"기준 모델 E1-01x2 (시드 0·1) 벌금 {b['penalty']:,.0f}원  (팀 기록 214,063,738원)")
print(f"  11~15시 벌금 비중 {b['peak'] / b['penalty'] * 100:.1f}%")
print(f"같은 설정, 시드만 2·3 (E1-01s2x2) 벌금 {s['penalty']:,.0f}원 → {pct(s['penalty'], b['penalty']):+.2f}%  (시드 잡음의 크기)")

기준 모델 E1-01x2 (시드 0·1) 벌금 214,063,738원  (팀 기록 214,063,738원)
  11~15시 벌금 비중 82.4%
같은 설정, 시드만 2·3 (E1-01s2x2) 벌금 214,488,953원 → +0.20%  (시드 잡음의 크기)


## 1라운드 — 팀 문서의 계획대로

역할 분담 문서의 E2-02 ~ E2-07을 그대로 시험했다. 근거는 EDA의 상관(맑은날 대비 발전율과 습도 −0.57, 운량 −0.53, 운고 +0.50, 강수확률 −0.45)이었다. E2-05b(직전 시각 자외선지수)는 팀 문서에 없는 추가 실험이다. 다른 파이프라인에서 이 변수가 중요도 1위였기 때문에 넣어 봤다.

In [2]:
R1 = [("E2-02x2", "운고 wb_ceiling"),
      ("E2-03x2", "운량·습도의 두 발행분 평균과 차이"),
      ("E2-04x2", "하루 요약: 해 뜬 시간 평균 운량(A)·일교차(A)·하루 강수 합(B)"),
      ("E2-05x2", "운량 1시간 전·후, 3시간 평균"),
      ("E2-05bx2", "자외선지수 1시간 전·후 (추가 실험)"),
      ("E2-06x2", "같은 시각 전 발전소 평균 운량(A·B)"),
      ("E2-07x2", "A 운량 − B 운량과 그 절댓값")]
table(R1)

,바꾼 것,벌금(원),E1-01x2 대비,11~15시 대비,나아진 달,판정
실험,,,,,,
E2-02x2,운고 wb_ceiling,"214,141,791",+0.04%,+0.24%,6/12,잡음 범위
E2-03x2,운량·습도의 두 발행분 평균과 차이,"213,885,743",-0.08%,-0.05%,5/12,잡음 범위
E2-04x2,하루 요약: 해 뜬 시간 평균 운량(A)·일교차(A)·하루 강수 합(B),"209,676,101",-2.05%,-1.78%,8/12,채택 기준 충족
E2-05x2,"운량 1시간 전·후, 3시간 평균","213,992,844",-0.03%,+0.16%,6/12,잡음 범위
E2-05bx2,자외선지수 1시간 전·후 (추가 실험),"214,972,282",+0.42%,+0.65%,4/12,잡음 범위
E2-06x2,같은 시각 전 발전소 평균 운량(A·B),"213,591,756",-0.22%,-0.14%,7/12,잡음 범위
E2-07x2,A 운량 − B 운량과 그 절댓값,"214,408,373",+0.16%,+0.11%,6/12,잡음 범위


**판단**
- 기준을 넘은 것은 **E2-04(하루 요약) 하나**다(−2.05%, 11~15시 −1.78%).
- 운고(E2-02)는 상관이 컸지만 효과가 없었다. 운량·습도에 이미 같은 정보가 들어 있는 것으로 본다.
- 시각 단위 피처(앞뒤 시각, 전 지점 평균, 제공사 차이, 라운드 요약)는 모두 ±0.4% 안이다. 한 시각의 예보를 다르게 표현하는 것은 모델이 이미 하고 있는 일이라는 해석이다.
- 직전 시각 자외선지수(E2-05b)는 +0.42%로 오히려 나빴다. 다른 모델의 "중요도 1위"는 그 변수가 꼭 필요하다는 뜻이 아니었다.
- → 다음 질문: **하루 요약의 어떤 요소가 효과를 내는가?**

## 2라운드 — 하루 요약을 쪼개고 넓히기

E2-04의 세 요소를 하나씩만 넣어 보고(E2-04a·b·c), 하루 요약에 다른 하루 변수를 더해 봤다(E2-08·09·10).

In [3]:
R2 = [("E2-04x2", "(비교) 하루 요약 3개"),
      ("E2-04ax2", "하루 평균 운량만"),
      ("E2-04bx2", "일교차만"),
      ("E2-04cx2", "하루 강수 합만"),
      ("E2-08x2", "E2-04 + B 하루 운량·A 하루 습도·B 강수확률·B 자외선 최대"),
      ("E2-09x2", "E2-04 + 오전·오후 평균 운량"),
      ("E2-10x2", "E2-04 + 시각 운량의 하루 평균 대비 차이(A·B)")]
table(R2)

,바꾼 것,벌금(원),E1-01x2 대비,11~15시 대비,나아진 달,판정
실험,,,,,,
E2-04x2,(비교) 하루 요약 3개,"209,676,101",-2.05%,-1.78%,8/12,채택 기준 충족
E2-04ax2,하루 평균 운량만,"212,964,069",-0.51%,-0.47%,7/12,채택 기준 충족
E2-04bx2,일교차만,"212,595,429",-0.69%,-0.43%,7/12,채택 기준 충족
E2-04cx2,하루 강수 합만,"214,472,129",+0.19%,+0.27%,7/12,잡음 범위
E2-08x2,E2-04 + B 하루 운량·A 하루 습도·B 강수확률·B 자외선 최대,"211,157,656",-1.36%,-1.10%,8/12,채택 기준 충족
E2-09x2,E2-04 + 오전·오후 평균 운량,"209,944,857",-1.92%,-1.41%,7/12,채택 기준 충족
E2-10x2,E2-04 + 시각 운량의 하루 평균 대비 차이(A·B),"212,344,840",-0.80%,-0.31%,6/12,채택 기준 충족


**판단**
- 하나씩 넣으면 운량 −0.51%, 일교차 −0.69%, 강수 합 +0.19%다. 셋을 함께 넣은 E2-04(−2.05%)가 합보다 크다. 세 값을 같이 봐야 "그날이 어떤 날인지"가 드러난다는 뜻으로 읽힌다(맑고 건조한 날은 일교차가 크고 비가 없다).
- 다른 하루 변수를 더하면 모두 E2-04보다 나빠졌다(E2-08 −1.36%, E2-10 −0.80%). E2-09(오전·오후)는 −1.92%로 E2-04와 잡음 범위 안이라 이득이 없다.

## 3라운드 — 빼기와 쌓기

강수 합을 뺀 버전(E2-11)을 기준 모델 대비로, E2-04 위에 1라운드에서 약하게 좋았던 두 개를 얹은 버전(E2-12·13)을 **E2-04 대비**로 본다.

In [4]:
display(table([("E2-11x2", "하루 평균 운량 + 일교차 (강수 합 뺌)")]))
table([("E2-12x2", "E2-04 + 전 발전소 평균 운량"),
       ("E2-13x2", "E2-04 + 두 발행분 평균·차이")], ref="E2-04x2")

,바꾼 것,벌금(원),E1-01x2 대비,11~15시 대비,나아진 달,판정
실험,,,,,,
E2-11x2,하루 평균 운량 + 일교차 (강수 합 뺌),"212,255,290",-0.84%,-0.57%,7/12,채택 기준 충족


,바꾼 것,벌금(원),E2-04x2 대비,11~15시 대비,나아진 달,판정
실험,,,,,,
E2-12x2,E2-04 + 전 발전소 평균 운량,"210,214,015",+0.26%,+0.23%,5/12,잡음 범위
E2-13x2,E2-04 + 두 발행분 평균·차이,"209,740,385",+0.03%,-0.04%,6/12,잡음 범위


**판단**
- 강수 합을 빼면 −0.84%로, E2-04(−2.05%)의 절반에 못 미친다. 시드 2·3에서도 강수 합을 뺀 쪽이 덜 줄었다(시드 절 참고). 혼자서는 효과가 없지만 다른 두 값과 함께일 때 쓸모가 있어서 남긴다.
- E2-04에 다른 것을 얹으면 개선이 없거나(+0.03%) 오히려 조금 나빠진다(+0.26%).

## 4라운드 — 하루 요약을 다르게 계산하기

E2-14는 두 발행분(10시·17시)의 평균값으로 하루 요약을 냈다. E2-15는 B사 값으로도 하루 요약을 만들어 더했다(B 하루 운량, B 일교차, A 하루 강수 합). 제공사마다 예보 오차가 달라서, 두 쪽을 함께 보면 "그날"을 더 정확히 짚을 수 있다는 가설이다.

In [5]:
table([("E2-04x2", "(비교) 하루 요약 3개, A·B 섞어서"),
       ("E2-14x2", "하루 요약을 두 발행분 평균값으로"),
       ("E2-15x2", "E2-04 + B 하루 운량 + B 일교차 + A 하루 강수 합")])

,바꾼 것,벌금(원),E1-01x2 대비,11~15시 대비,나아진 달,판정
실험,,,,,,
E2-04x2,"(비교) 하루 요약 3개, A·B 섞어서","209,676,101",-2.05%,-1.78%,8/12,채택 기준 충족
E2-14x2,하루 요약을 두 발행분 평균값으로,"214,412,243",+0.16%,+0.67%,5/12,잡음 범위
E2-15x2,E2-04 + B 하루 운량 + B 일교차 + A 하루 강수 합,"207,428,310",-3.10%,-3.03%,9/12,채택 기준 충족


**판단**
- 발행분 평균(E2-14)은 +0.16%로 효과가 사라졌다. 17시 발행분이 더 최신이라, 10시 발행분과 섞으면 정보가 흐려지는 것으로 본다.
- 두 제공사 버전(E2-15)은 −3.10%, 11~15시 −3.03%로 E2-04보다 1%p 이상 낮다.

## 5라운드 — 두 제공사 버전 다듬기 (중간 결론)

E2-08에서 B 하루 운량을 넣었을 때 오히려 나빴다. 그래서 E2-15에서 B 하루 운량을 빼 봤다(E2-16). 이슬점 차이를 더해 보기도 했다(E2-17). 둘 다 **E2-15 대비**로 본다.

In [6]:
table([("E2-16x2", "E2-15에서 B 하루 운량 뺌 (피처 5개)"),
       ("E2-17x2", "E2-15 + 해 뜬 시간 평균 이슬점 차이")], ref="E2-15x2")

,바꾼 것,벌금(원),E2-15x2 대비,11~15시 대비,나아진 달,판정
실험,,,,,,
E2-16x2,E2-15에서 B 하루 운량 뺌 (피처 5개),"206,704,167",-0.35%,-0.22%,7/12,잡음 범위
E2-17x2,E2-15 + 해 뜬 시간 평균 이슬점 차이,"208,009,515",+0.28%,+0.48%,4/12,잡음 범위


**판단**: B 하루 운량을 빼도 나빠지지 않고(E2-16, E2-15 대비 −0.35%), 이슬점 차이는 나빠진다. 여기까지의 중간 결론은 **E2-16(기준 모델 대비 −3.44%)** 이었다. 이후 라운드는 E2-16 위에서 더 줄일 수 있는지를 본다.

## 6라운드 — E2-16 위에 하루 정보 하나씩 더하기

"그날의 성격"을 더 정확히 알려줄 만한 값을 하나씩 얹었다(**E2-16 대비**).
- E2-18 운량 표준편차: 구름이 지나가는 날
- E2-19 비 온 시간 수: 비가 오래 온 날
- E2-20 해 뜬 시간 일교차: 밤 기온 잡음 제거, 교체
- E2-21 전날 강수 합: 전 대상일 예보는 예측시점보다 먼저 나와 쓸 수 있다
- E2-22 전 발전소 하루 평균: 지역 전체가 맑은 날인지
- E2-23 강수확률: E2-08에서 다른 것과 묶여 따로 보지 못했다

In [7]:
table([("E2-18x2", "+ 해 뜬 시간 운량 표준편차"),
       ("E2-19x2", "+ 비 온 시간 수 (A·B)"),
       ("E2-20x2", "일교차를 해 뜬 시간으로만 (교체)"),
       ("E2-21x2", "+ 전날 강수 합 (A·B)"),
       ("E2-22x2", "+ 전 발전소 하루 평균 (운량·일교차·강수)"),
       ("E2-23x2", "+ 해 뜬 시간 평균 강수확률 (B)")], ref="E2-16x2")

,바꾼 것,벌금(원),E2-16x2 대비,11~15시 대비,나아진 달,판정
실험,,,,,,
E2-18x2,+ 해 뜬 시간 운량 표준편차,"206,194,643",-0.25%,-0.40%,8/12,잡음 범위
E2-19x2,+ 비 온 시간 수 (A·B),"207,100,773",+0.19%,+0.06%,6/12,잡음 범위
E2-20x2,일교차를 해 뜬 시간으로만 (교체),"206,069,846",-0.31%,-0.42%,7/12,잡음 범위
E2-21x2,+ 전날 강수 합 (A·B),"202,444,653",-2.06%,-2.21%,9/12,채택 기준 충족
E2-22x2,+ 전 발전소 하루 평균 (운량·일교차·강수),"209,531,507",+1.37%,+1.64%,5/12,악화
E2-23x2,+ 해 뜬 시간 평균 강수확률 (B),"206,634,358",-0.03%,-0.16%,6/12,잡음 범위


**판단**
- **전날 강수 합(E2-21)이 E2-16 대비 −2.06%로 크게 좋아졌다.** 나머지는 잡음 범위이거나(E2-18·19·20·23) 나빠졌다(E2-22, +1.37%).
- 전날(D−1) 예보는 D−2일 17시에 발행된다. D일을 예측하는 시점(D−1일 17시)에 이미 있는 정보라 규정 위반이 아니다.
- 왜 도움이 되는지에 대한 해석: 비를 몰고 온 기상 시스템은 하루 만에 사라지지 않는다. 전날 비가 왔다는 정보가 "오늘 예보가 맑다고 해도 실제로는 덜 맑을 수 있다"는 신호가 되는 것으로 본다. 비가 패널을 씻어 주는 효과도 있을 수 있지만, 이 데이터로는 둘을 구분하지 못한다.
- → 전날 정보를 더 넓혀 본다.

## 7라운드 — 전날 정보 넓히기 (E2-21 위에)

전날 운량·일교차를 더하기(E2-24), 이틀 전 강수 합을 더하기(E2-25), 일교차를 해 뜬 시간으로 바꾸기(E2-26)를 시험했다(**E2-21 대비**).

In [8]:
table([("E2-24x2", "+ 전날 평균 운량·일교차 (A)"),
       ("E2-25x2", "+ 이틀 전 강수 합 (A·B)"),
       ("E2-26x2", "일교차를 해 뜬 시간으로만 (교체)")], ref="E2-21x2")

,바꾼 것,벌금(원),E2-21x2 대비,11~15시 대비,나아진 달,판정
실험,,,,,,
E2-24x2,+ 전날 평균 운량·일교차 (A),"199,632,154",-1.39%,-1.64%,9/12,채택 기준 충족
E2-25x2,+ 이틀 전 강수 합 (A·B),"201,674,928",-0.38%,-0.53%,6/12,잡음 범위
E2-26x2,일교차를 해 뜬 시간으로만 (교체),"202,657,331",+0.11%,+0.16%,4/12,잡음 범위


**판단**: 전날 운량·일교차를 더한 **E2-24가 E2-21 대비 −1.39%** 로 가장 좋다(기준 모델 대비 −6.74%). 이틀 전 강수는 잡음 범위이고, 해 뜬 시간 일교차는 이번에도 이득이 없다.

## 8라운드 — 착시 점검: 실적이 빠진 행 때문은 아닌가

지금까지의 하루 요약·전날 값은 **그 발전소에 실적이 있는 행**으로 계산한다. 전날 데이터가 통째로 빠진 날(통신 장애, 편입 첫날)에는 전날 피처가 빈다. "실적이 빠졌다"는 사실은 정답 쪽 정보라 새어 들어갈 수 있다. 실제 평가 기간은 모든 시각이 다 있어서 이런 빈칸이 생기지 않는다.

이를 두 가지로 확인했다.
1. 검증 예측 행에서 이 피처들이 비는 비율
2. 같은 피처를 **관측지점 예보 자체**로 계산한 버전(E2-27): 여러 발전소 행에서 (관측지점, 시각)별 예보를 한 번씩 모아 계산하므로 한 발전소의 실적 유무와 무관하다. E2-28은 여기에 전날 B사 일교차를 더한 것이다.

In [9]:
import train_site_model as tsm
fr = tsm.load_frame("valid", 2)
fr.loc[~fr.is_fit, ["energy", "cf"]] = np.nan
a = fw.add_e21_prev_more(fr.copy())
st = fw.add_station_day(fr.copy())
day = a.elev > 0
prev_cols = ["wx_prev_rain", "wx_prev_rain_a", "wx_prev_cloud", "wx_prev_trange"]
print("E2-24 전날 피처 결측 비율 (해 뜬 행)")
for part, m in [("  학습 행     ", a.is_fit & day), ("  검증 예측 행", ~a.is_fit & day)]:
    print(part, ", ".join(f"{c} {a.loc[m, c].isna().mean() * 100:.2f}%" for c in prev_cols))
pairs = [("wx_day_cloud", "wx_day_cloud"), ("wx_day_rain", "wx_day_rain"), ("wx_prev_rain", "wx_day_rain_prev"),
         ("wx_prev_cloud", "wx_day_cloud_prev"), ("wx_prev_trange", "wx_day_trange_prev")]
print("발전소 행 기준(E2-24) vs 관측지점 기준(E2-27) 값 비교 (둘 다 있는 해 뜬 행)")
for x, y in pairs:
    both = a[x].notna() & st[y].notna() & day
    diff = (a.loc[both, x] - st.loc[both, y]).abs()
    print(f"  {x:<15} 값이 다른 행 {(diff > 1e-3).mean() * 100:.2f}%, 평균 절대차 {diff.mean():.4f}")
del fr, a, st

E2-24 전날 피처 결측 비율 (해 뜬 행)
  학습 행      wx_prev_rain 0.17%, wx_prev_rain_a 0.17%, wx_prev_cloud 0.19%, wx_prev_trange 0.17%
  검증 예측 행 wx_prev_rain 0.03%, wx_prev_rain_a 0.03%, wx_prev_cloud 0.03%, wx_prev_trange 0.03%
발전소 행 기준(E2-24) vs 관측지점 기준(E2-27) 값 비교 (둘 다 있는 해 뜬 행)


  wx_day_cloud    값이 다른 행 0.61%, 평균 절대차 0.0127
  wx_day_rain     값이 다른 행 0.01%, 평균 절대차 0.0003


  wx_prev_rain    값이 다른 행 0.02%, 평균 절대차 0.0007
  wx_prev_cloud   값이 다른 행 0.65%, 평균 절대차 0.0156


  wx_prev_trange  값이 다른 행 0.07%, 평균 절대차 0.0020


In [10]:
table([("E2-24x2", "(비교) 발전소 행 기준"),
       ("E2-27x2", "E2-24와 같은 피처를 관측지점 기준으로"),
       ("E2-28x2", "E2-27 + 전날 B사 일교차")])

,바꾼 것,벌금(원),E1-01x2 대비,11~15시 대비,나아진 달,판정
실험,,,,,,
E2-24x2,(비교) 발전소 행 기준,"199,632,154",-6.74%,-6.93%,11/12,채택 기준 충족
E2-27x2,E2-24와 같은 피처를 관측지점 기준으로,"201,155,364",-6.03%,-6.13%,9/12,채택 기준 충족
E2-28x2,E2-27 + 전날 B사 일교차,"200,456,473",-6.36%,-6.47%,12/12,채택 기준 충족


**판단**
- 전날 피처가 비는 행은 검증 예측 행의 0.03%뿐이다. 이걸로 1~2%의 개선을 설명할 수 없다.
- 관측지점 기준으로 다시 계산한 값은 발전소 행 기준 값과 99% 이상 같다(운량만 0.6% 정도의 행에서 조금 다르다. 해 뜬 시각 판정이 발전소 위치마다 조금 달라서다). 별도 진단에서 평가 기간 행의 결측은 0%였다.
- E2-27·28도 기준 모델 대비 −6%대다. **전날 정보의 효과는 실적 결측이 만든 착시가 아니다.**
- 셋 사이의 차이(0.3 ~ 0.7%p)는 시드 잡음과 비슷한 크기라, 아래 시드 4개와 백테스트로 가린다.

## 시드 강건성 — 순위가 시드 운이 아닌가

후보마다 시드 0·1과 시드 2·3을 각각 돌렸다. 같은 시드의 기준 모델과 비교하고, 두 결과를 평균한 "시드 4개" 예측으로도 채점했다.

In [11]:
def avg_preds(*names):
    ps = [read_preds(PREDS / f"{n}.parquet").set_index(["site_id", "ts"]).pred_kwh for n in names]
    return (sum(ps) / len(ps)).reset_index()

champ4 = evaluate_preds(avg_preds("E1-01x2", "E1-01s2x2"), actual=ACTUAL)
CANDS = [("E2-04", "하루 요약 3개"), ("E2-11", "운량+일교차"), ("E2-15", "두 제공사 6개"), ("E2-16", "두 제공사 5개"),
         ("E2-21", "E2-16 + 전날 강수"), ("E2-24", "E2-21 + 전날 운량·일교차"),
         ("E2-27", "E2-24 관측지점 기준"), ("E2-28", "E2-27 + 전날 B 일교차"),
         ("E2-40", "E2-24 + 아침 안개 신호 (11라운드)")]
seed4 = {}
rows = []
for n, desc in CANDS:
    a, c = score(f"{n}x2"), score(f"{n}s2x2")
    four = evaluate_preds(avg_preds(f"{n}x2", f"{n}s2x2"), actual=ACTUAL)
    seed4[n] = pct(four["penalty"], champ4["penalty"])
    rows.append({"후보": f"{n} {desc}",
                 "시드 0·1": f"{pct(a['penalty'], score('E1-01x2')['penalty']):+.2f}%",
                 "시드 2·3": f"{pct(c['penalty'], score('E1-01s2x2')['penalty']):+.2f}%",
                 "시드 4개 벌금(원)": f"{four['penalty']:,.0f}",
                 "시드 4개": f"{seed4[n]:+.2f}%",
                 "11~15시(시드 4개)": f"{pct(four['peak'], champ4['peak']):+.2f}%",
                 "나아진 달(시드 4개)": f"{int((four['by_month'] < champ4['by_month']).sum())}/12"})
print(f"기준 모델 시드 4개 벌금 {champ4['penalty']:,.0f}원")
pd.DataFrame(rows).set_index("후보")

기준 모델 시드 4개 벌금 213,975,269원


,시드 0·1,시드 2·3,시드 4개 벌금(원),시드 4개,11~15시(시드 4개),나아진 달(시드 4개)
후보,,,,,,
E2-04 하루 요약 3개,-2.05%,-1.47%,"210,149,058",-1.79%,-1.54%,7/12
E2-11 운량+일교차,-0.84%,-1.01%,"212,021,579",-0.91%,-0.66%,6/12
E2-15 두 제공사 6개,-3.10%,-3.46%,"206,909,656",-3.30%,-3.18%,8/12
E2-16 두 제공사 5개,-3.44%,-3.17%,"206,836,413",-3.34%,-3.14%,8/12
E2-21 E2-16 + 전날 강수,-5.43%,-5.98%,"201,723,489",-5.73%,-5.70%,10/12
E2-24 E2-21 + 전날 운량·일교차,-6.74%,-6.24%,"200,016,701",-6.52%,-6.61%,11/12
E2-27 E2-24 관측지점 기준,-6.03%,-6.69%,"200,332,679",-6.38%,-6.42%,10/12
E2-28 E2-27 + 전날 B 일교차,-6.36%,-5.68%,"201,031,669",-6.05%,-6.05%,11/12
E2-40 E2-24 + 아침 안개 신호 (11라운드),-6.98%,-6.65%,"199,353,957",-6.83%,-6.83%,11/12


**판단**
- 개선 폭의 큰 흐름(E2-11 < E2-04 < E2-15 ≈ E2-16 < 전날 정보를 쓴 E2-21·24·27·28)은 시드 0·1, 시드 2·3, 시드 4개에서 모두 같다. 하루 요약과 전날 정보의 효과는 시드 운이 아니다.
- 전날 정보를 쓴 네 후보 안에서는 순위가 시드마다 바뀐다. 이 시점(8라운드)에서는 시드 4개 기준 E2-24가 가장 좋았다(−6.52%, 12개월 중 11개월 개선).
- 표 맨 아래 E2-40은 뒤의 11라운드에서 찾은 최종안이다(시드 4개 −6.83%). 비교를 위해 같은 표에 넣었다.

## 다른 해 백테스트 — 그해에만 맞는 것을 고르지 않았나

31개 조합을 같은 가상 평가연도로 비교했다. 그러니 그해 날씨에만 잘 맞는 것을 골랐을 위험이 있다. 그래서 학습 컷오프를 1년 앞으로 옮겨(2021-06 ~ 2023-05 학습 → 2023-06 ~ 2024-05 예측) 같은 방식으로 다시 채점했다. 학습 방식과 채점식은 팀 것을 그대로 썼다(`analysis_lab/E2_backtest.py`, 공용 스크립트는 고치지 않음). 그해에는 발전소가 늘어나는 중이라 실적이 있는 205곳으로 채점된다. 시드는 0·1이다.

In [12]:
bt = pd.read_csv(ROOT / "analysis_lab" / "E2_backtest_results.csv").set_index("config")
show = bt.loc[["E1-01", "E2-04", "E2-16", "E2-21", "E2-24", "E2-27", "E2-28"]].copy()
show["penalty"] = show.penalty.map("{:,.0f}".format)
show["vs_E1-01_%"] = show["vs_E1-01_%"].map("{:+.2f}%".format)
show["peak_vs_E1-01_%"] = show["peak_vs_E1-01_%"].map("{:+.2f}%".format)
show.columns = ["벌금(원)", "11~15시 벌금(원)", "기준 대비", "11~15시 대비", "나아진 달"]
show["11~15시 벌금(원)"] = bt.loc[show.index, "peak"].map("{:,.0f}".format)
show

,벌금(원),11~15시 벌금(원),기준 대비,11~15시 대비,나아진 달
config,,,,,
E1-01,"113,829,918","94,439,392",+0.00%,+0.00%,0
E2-04,"110,788,841","92,172,404",-2.67%,-2.40%,8
E2-16,"108,484,867","90,028,322",-4.70%,-4.67%,10
E2-21,"106,468,109","88,466,060",-6.47%,-6.33%,11
E2-24,"106,598,935","88,488,336",-6.35%,-6.30%,12
E2-27,"107,700,064","89,736,590",-5.39%,-4.98%,12
E2-28,"105,810,339","87,891,437",-7.05%,-6.93%,12


In [13]:
summ = pd.DataFrame({"가상 평가연도 시드 4개": {n: seed4[n] for n in ["E2-04", "E2-16", "E2-21", "E2-24", "E2-27", "E2-28"]},
                     "다른 해 백테스트": bt.loc[["E2-04", "E2-16", "E2-21", "E2-24", "E2-27", "E2-28"], "vs_E1-01_%"]})
summ["두 해 평균"] = summ.mean(axis=1)
summ["시드 0·1"] = [pct(score(f"{n}x2")["penalty"], score("E1-01x2")["penalty"]) for n in summ.index]
summ["시드 2·3"] = [pct(score(f"{n}s2x2")["penalty"], score("E1-01s2x2")["penalty"]) for n in summ.index]
summ["측정 3개의 폭(최대−최소)"] = summ[["시드 0·1", "시드 2·3", "다른 해 백테스트"]].max(axis=1) - summ[["시드 0·1", "시드 2·3", "다른 해 백테스트"]].min(axis=1)
summ.round(2)

,가상 평가연도 시드 4개,다른 해 백테스트,두 해 평균,시드 0·1,시드 2·3,측정 3개의 폭(최대−최소)
E2-04,-1.79,-2.67,-2.23,-2.05,-1.47,1.20
E2-16,-3.34,-4.70,-4.02,-3.44,-3.17,1.52
E2-21,-5.73,-6.47,-6.10,-5.43,-5.98,1.04
E2-24,-6.52,-6.35,-6.44,-6.74,-6.24,0.50
E2-27,-6.38,-5.39,-5.88,-6.03,-6.69,1.30
E2-28,-6.05,-7.05,-6.55,-6.36,-5.68,1.36


**판단**
- 다른 해에서도 순서가 같다. 하루 요약(E2-04 −2.67%) < 두 제공사(E2-16 −4.70%) < 전날 정보(E2-21·24·27·28, −5.4 ~ −7.1%). 고른 피처가 2024-25년에만 맞는 것은 아니다.
- **이 시점에서는 E2-24를 골랐다**(이후 11라운드에서 E2-40으로 바뀐다).
  - 가상 평가연도 시드 4개에서 1위다(−6.52%).
  - 세 번의 측정(시드 0·1, 시드 2·3, 다른 해)이 −6.2 ~ −6.7%로 가장 고르다.
  - E2-28은 두 해 평균이 비슷하지만 측정마다 −5.7 ~ −7.1%로 흔들리고 피처도 하나 더 많다.
  - E2-27(관측지점 기준)은 다른 해에서 −5.39%로 가장 약했다.

## 9라운드 — E2-24 위에 새 정보 더하기

전날 정보가 통했으니 "날씨가 바뀌는 날"을 알려줄 만한 값을 더 얹어 봤다(**E2-24 대비**).
- E2-29 오늘 − 전날 변화량(운량·일교차)
- E2-30 해 뜬 시간 평균 습도(오늘·전날): EDA 상관 1위(−0.57)였지만 단독으로는 시험하지 않았다
- E2-31 하루 평균 기압과 전날 대비 변화: 전선 통과를 알려주는 고전적 신호
- E2-32 하루 평균 풍속(오늘·전날)
- E2-33 강수 합을 해 뜬 시간만으로, 교체: 밤비는 발전과 무관할 수 있다

In [14]:
table([("E2-29x2", "+ 오늘 − 전날 (운량·일교차)"),
       ("E2-30x2", "+ 해 뜬 시간 평균 습도 (오늘·전날)"),
       ("E2-31x2", "+ 하루 평균 기압과 전날 대비 변화"),
       ("E2-32x2", "+ 하루 평균 풍속 (오늘·전날)"),
       ("E2-33x2", "강수 합을 해 뜬 시간만으로 (교체)")], ref="E2-24x2")

,바꾼 것,벌금(원),E2-24x2 대비,11~15시 대비,나아진 달,판정
실험,,,,,,
E2-29x2,+ 오늘 − 전날 (운량·일교차),"201,423,475",+0.90%,+1.14%,4/12,악화
E2-30x2,+ 해 뜬 시간 평균 습도 (오늘·전날),"200,415,887",+0.39%,+0.38%,6/12,잡음 범위
E2-31x2,+ 하루 평균 기압과 전날 대비 변화,"201,485,475",+0.93%,+1.06%,5/12,악화
E2-32x2,+ 하루 평균 풍속 (오늘·전날),"202,017,523",+1.19%,+1.50%,3/12,악화
E2-33x2,강수 합을 해 뜬 시간만으로 (교체),"203,003,623",+1.69%,+1.78%,4/12,악화


**판단**
- 다섯 개 모두 E2-24보다 나빴다(+0.39 ~ +1.69%).
- 변화량(E2-29)은 모델이 오늘 값과 전날 값으로 이미 계산할 수 있는 정보라 이득이 없다.
- 습도·기압·풍속은 운량·일교차·강수에 이미 담긴 "그날의 성격"과 겹친다.
- 밤비를 뺀 강수(E2-33)가 오히려 나쁜 것은, 밤에 내린 비도 다음 날 아침 구름·습도와 이어지기 때문으로 본다.
- 이 단계부터는 **더할수록 나빠지는 흐름**이 뚜렷하다. 그렇다면 오히려 덜어내면 좋아질 수도 있다 → 10라운드.

## 10라운드 — E2-24에서 덜어내기

E2-24의 9개를 성격별 묶음으로 나눠 하나씩 빼 봤다(**E2-24 대비**). 성능이 같다면 피처가 적은 쪽이 실제 평가에서도 더 안전하다.

In [15]:
table([("E2-34x2", "A사 강수 합 빼기 (오늘·전날)"),
       ("E2-35x2", "B사 일교차 빼기"),
       ("E2-36x2", "전날 운량 빼기"),
       ("E2-37x2", "전날 일교차 빼기")], ref="E2-24x2")

,바꾼 것,벌금(원),E2-24x2 대비,11~15시 대비,나아진 달,판정
실험,,,,,,
E2-34x2,A사 강수 합 빼기 (오늘·전날),"204,970,303",+2.67%,+3.00%,1/12,악화
E2-35x2,B사 일교차 빼기,"203,574,327",+1.97%,+2.27%,3/12,악화
E2-36x2,전날 운량 빼기,"201,872,289",+1.12%,+1.47%,2/12,악화
E2-37x2,전날 일교차 빼기,"200,702,066",+0.54%,+0.61%,4/12,악화


**판단**
- 어느 묶음을 빼도 나빠졌다. 영향이 큰 순서는 A사 강수(+2.67%), B사 일교차(+1.97%), 전날 운량(+1.12%), 전날 일교차(+0.54%)다.
- 두 제공사의 강수·일교차를 함께 보는 것이 특히 중요하다. 한 제공사만 쓰면 그 회사 예보가 빗나간 날을 잡지 못한다.
- 전날 일교차를 뺀 E2-37만 잡음 경계(+0.54%)에 있어서, 습도를 더한 E2-30(+0.39%)과 함께 최종 검증에 넣었다.

## 중간 검증 — E2-24와 가장 가까운 두 후보, 측정 4개

E2-24, E2-30(+습도, 11개), E2-37(−전날 일교차, 8개)을 네 가지로 쟀다. 가상 평가연도와 다른 해, 각각 시드 0·1과 2·3이다. 모든 값은 **같은 해·같은 시드의 기준 모델 대비**다.

In [16]:
bt = pd.read_csv(ROOT / "analysis_lab" / "E2_backtest_results.csv").set_index("config")
fin_rows = []
for n, desc in [("E2-24", "최종 (9개)"), ("E2-30", "+ 습도 (11개)"), ("E2-37", "− 전날 일교차 (8개)")]:
    v = {"가상 평가연도 시드 0·1": pct(score(f"{n}x2")["penalty"], score("E1-01x2")["penalty"]),
         "가상 평가연도 시드 2·3": pct(score(f"{n}s2x2")["penalty"], score("E1-01s2x2")["penalty"]),
         "다른 해 시드 0·1": bt.loc[n, "vs_E1-01_%"],
         "다른 해 시드 2·3": bt.loc[f"{n}@s2", "vs_E1-01_%"]}
    vals = list(v.values())
    fin_rows.append({"후보": f"{n} {desc}", **{k: round(x, 2) for k, x in v.items()},
                     "평균": round(float(np.mean(vals)), 2), "폭(최대−최소)": round(max(vals) - min(vals), 2)})
pd.DataFrame(fin_rows).set_index("후보")

,가상 평가연도 시드 0·1,가상 평가연도 시드 2·3,다른 해 시드 0·1,다른 해 시드 2·3,평균,폭(최대−최소)
후보,,,,,,
E2-24 최종 (9개),-6.74,-6.24,-6.35,-5.44,-6.19,1.30
E2-30 + 습도 (11개),-6.38,-6.51,-6.08,-5.56,-6.13,0.96
E2-37 − 전날 일교차 (8개),-6.24,-6.10,-6.19,-5.89,-6.10,0.35


**판단**
- 세 후보의 평균 차이는 0.1%p 안이다. 측정 하나하나에서는 1위가 바뀐다(시드 잡음 ±0.5%보다 작은 차이).
- 이 시점에서는 **E2-24를 유지했다.** 평균이 가장 좋고(−6.19%), 가상 평가연도 시드 4개 평균에서도 1위였다.
- E2-37은 피처가 하나 적고 측정 간 흔들림이 가장 작다(폭 0.35%p). E2-30은 피처가 둘 많은데 나아지지 않았다.

## 11라운드 — 아직 쓰지 않은 정보

지금까지 결과에서 단서를 얻어 가설을 세웠다(**E2-24 대비**).
- E2-38 강설 합(오늘·전날, A·B): E2-24에서 유일하게 나빠진 달이 1월이다. 패널에 쌓인 눈은 며칠씩 발전을 막는다
- E2-39 밤사이 강수(전날 19시 ~ 오늘 06시, A·B): 하루 합계보다 아침 구름과 더 직접 이어질 수 있다
- E2-40 아침 안개 신호(06~10시 최저 시정 B, 06~10시 평균 습도 A): 아침 안개·박무는 오전 발전을 깎는다
- E2-41 주변 발전소 10곳의 하루 요약 평균: 관측지점 한 곳 예보의 잡음 줄이기
- E2-42 시각별 풍속·기압 예보를 빈 값으로(빼는 효과 확인). 하루 요약이 생긴 지금은 잡음일 수 있다

In [17]:
table([("E2-38x2", "+ 강설 합 (오늘·전날, A·B)"),
       ("E2-39x2", "+ 밤사이 강수 (A·B)"),
       ("E2-40x2", "+ 아침 안개 신호 (최저 시정 B, 아침 습도 A)"),
       ("E2-41x2", "+ 주변 발전소 10곳 하루 요약 평균"),
       ("E2-42x2", "시각별 풍속·기압 예보 빼기")], ref="E2-24x2")

,바꾼 것,벌금(원),E2-24x2 대비,11~15시 대비,나아진 달,판정
실험,,,,,,
E2-38x2,"+ 강설 합 (오늘·전날, A·B)","200,656,204",+0.51%,+0.58%,5/12,악화
E2-39x2,+ 밤사이 강수 (A·B),"200,634,996",+0.50%,+0.64%,3/12,악화
E2-40x2,"+ 아침 안개 신호 (최저 시정 B, 아침 습도 A)","199,117,580",-0.26%,-0.17%,5/12,잡음 범위
E2-41x2,+ 주변 발전소 10곳 하루 요약 평균,"200,448,157",+0.41%,+0.64%,4/12,잡음 범위
E2-42x2,시각별 풍속·기압 예보 빼기,"202,830,956",+1.60%,+1.81%,2/12,악화


**판단**
- E2-24를 이긴 것은 **아침 안개 신호(E2-40) 하나**다(E2-24 대비 −0.26%, 기준 모델 대비 −6.98%로 지금까지 최고).
- 이 차이는 시드 잡음 범위 안이다. 그래서 측정 4개로 따로 검증한다(아래).
- 강설·밤사이 강수·주변 발전소는 나빠졌다(+0.4 ~ +0.5%). 강설은 1월을 겨냥했지만 연간으로는 손해다.
- 시각별 풍속·기압을 빼면 +1.60% 나빠진다. 하루 요약이 있어도 이 변수들은 제 몫을 한다.

## E2-40 검증 — 측정 4개로도 E2-24보다 좋은가

In [18]:
bt = pd.read_csv(ROOT / "analysis_lab" / "E2_backtest_results.csv").set_index("config")
rows40 = []
for n, desc in [("E2-24", "중간 결론 (9개)"), ("E2-40", "+ 아침 안개 신호 (11개)")]:
    v = {"가상 평가연도 시드 0·1": pct(score(f"{n}x2")["penalty"], score("E1-01x2")["penalty"]),
         "가상 평가연도 시드 2·3": pct(score(f"{n}s2x2")["penalty"], score("E1-01s2x2")["penalty"]),
         "다른 해 시드 0·1": bt.loc[n, "vs_E1-01_%"],
         "다른 해 시드 2·3": bt.loc[f"{n}@s2", "vs_E1-01_%"]}
    vals = list(v.values())
    rows40.append({"후보": f"{n} {desc}", **{k: round(x, 2) for k, x in v.items()},
                   "평균": round(float(np.mean(vals)), 2), "폭(최대−최소)": round(max(vals) - min(vals), 2)})
t40 = pd.DataFrame(rows40).set_index("후보")
wins = int((t40.iloc[1, :4] < t40.iloc[0, :4]).sum())
print(f"E2-40 이 E2-24 보다 좋은 측정: {wins}/4")
t40

E2-40 이 E2-24 보다 좋은 측정: 4/4


,가상 평가연도 시드 0·1,가상 평가연도 시드 2·3,다른 해 시드 0·1,다른 해 시드 2·3,평균,폭(최대−최소)
후보,,,,,,
E2-24 중간 결론 (9개),-6.74,-6.24,-6.35,-5.44,-6.19,1.3
E2-40 + 아침 안개 신호 (11개),-6.98,-6.65,-6.41,-6.38,-6.61,0.6


**판단**
- **E2-40이 네 측정 모두에서 E2-24보다 좋다.** 평균은 −6.61% 대 −6.19%로 0.42%p 낫고, 측정 간 흔들림도 더 작다(폭 0.60 대 1.30%p).
- 측정 하나하나의 차이는 시드 잡음 범위인 것도 있다. 하지만 두 해·두 시드 쌍에서 모두 같은 방향이라 우연으로 보기 어렵다.
- → **최종을 E2-40으로 바꾼다.** 안개 신호를 더 다듬을 수 있는지 12라운드에서 확인한다.

## 12라운드 — 안개 신호 쪼개고 다듬기 (E2-40 대비)

In [19]:
table([("E2-43x2", "안개 신호 중 최저 시정만"),
       ("E2-44x2", "안개 신호 중 아침 습도만"),
       ("E2-45x2", "E2-40 + 아침 최저 운고 (B)")], ref="E2-40x2")

,바꾼 것,벌금(원),E2-40x2 대비,11~15시 대비,나아진 달,판정
실험,,,,,,
E2-43x2,안개 신호 중 최저 시정만,"201,386,866",+1.14%,+1.19%,5/12,악화
E2-44x2,안개 신호 중 아침 습도만,"200,690,404",+0.79%,+0.93%,4/12,악화
E2-45x2,E2-40 + 아침 최저 운고 (B),"199,836,444",+0.36%,+0.60%,5/12,잡음 범위


**판단**
- 시정만(+1.14%), 습도만(+0.79%) 남기면 나빠진다. 둘 중 하나만으로는 E2-24보다도 못하다. 시정이 낮아도 습도가 낮으면 안개가 아니라 미세먼지·황사일 수 있고, 습도가 높아도 시정이 좋으면 안개가 아니다. 둘을 함께 봐야 안개를 가려낸다는 해석과 맞는다.
- 아침 최저 운고를 더하면 +0.36% 나빠진다.
- → 이 시점의 최종 후보는 **E2-40(11개)** 이다. 다만 같은 해들로 너무 많이 골랐기 때문에, 아래에서 단순화·마지막 시험·민감도 점검으로 다시 확인한다.

## 단순화 시도 — E2-46 (E2-40에서 전날 일교차 빼기, 10개)

E2-24에서 전날 일교차는 효과가 가장 작은 묶음이었다(10라운드, +0.54%). 성능이 같다면 피처가 적은 쪽이 실제 평가에서 더 안전하므로 빼 보았다.

In [20]:
rows46 = []
for n, desc in [("E2-40", "최종 후보 (11개)"), ("E2-46", "− 전날 일교차 (10개)")]:
    v = {"가상 평가연도 시드 0·1": pct(score(f"{n}x2")["penalty"], score("E1-01x2")["penalty"]),
         "가상 평가연도 시드 2·3": pct(score(f"{n}s2x2")["penalty"], score("E1-01s2x2")["penalty"]),
         "2023-24 시드 0·1": bt.loc[n, "vs_E1-01_%"],
         "2023-24 시드 2·3": bt.loc[f"{n}@s2", "vs_E1-01_%"]}
    rows46.append({"후보": f"{n} {desc}", **{k: round(x, 2) for k, x in v.items()}, "평균": round(float(np.mean(list(v.values()))), 2)})
pd.DataFrame(rows46).set_index("후보")

,가상 평가연도 시드 0·1,가상 평가연도 시드 2·3,2023-24 시드 0·1,2023-24 시드 2·3,평균
후보,,,,,
E2-40 최종 후보 (11개),-6.98,-6.65,-6.41,-6.38,-6.61
E2-46 − 전날 일교차 (10개),-6.58,-6.42,-6.93,-6.75,-6.67


**판단**: 두 해 측정 4개의 평균은 E2-46(−6.67%)과 E2-40(−6.61%)이 같은 수준이다. 그래서 단순화 여부는 아래 마지막 시험으로 정하기로 했다.

## 마지막 시험 — 한 번도 고르는 데 쓰지 않은 해 (2022-06 ~ 2023-05)

지금까지 2024-25년(가상 평가연도)과 2023-24년(백테스트)을 모두 후보를 고르는 데 썼다. 둘 다 이미 "본 데이터"라, 여기서 좋은 것은 그 해들에 우연히 맞았을 수 있다. 같은 해로 49개를 비교해 가장 좋은 것을 고르는 과정을 반복했기 때문이다. 그래서 **아직 한 번도 판단에 쓰지 않은 2022-23년**으로 마지막 시험을 했다. 학습은 같은 방식이다(컷오프 전 2년, 실제로는 데이터가 2021-01부터라 약 16개월).

**결과를 보기 전에 정한 판단 규칙**
1. E2-40이 이 해에 E2-24보다 0.5% 넘게 나쁘면, 마지막 안개 신호는 우연으로 보고 E2-24로 되돌린다.
2. E2-24가 E2-16보다 0.5% 넘게 나쁘면, 전날 정보도 같은 의심을 받는다.
3. 단순화(E2-46)는 이 해에서 E2-40보다 나쁘지 않을 때만 채택한다.
4. 각 단계가 기준 모델보다 0.5% 이상 낫지 않으면 "이 해에서는 효과 없음"으로 적는다.

In [21]:
bt = pd.read_csv(ROOT / "analysis_lab" / "E2_backtest_results.csv").set_index("config")
rows22 = []
for n, desc in [("E2-04", "하루 요약"), ("E2-16", "두 제공사"), ("E2-24", "+ 전날 정보"), ("E2-37", "E2-24 − 전날 일교차"),
                ("E2-40", "+ 아침 안개 (최종 후보)"), ("E2-46", "E2-40 − 전날 일교차")]:
    a, c = bt.loc[f"{n}@y2022", "vs_E1-01_%"], bt.loc[f"{n}@y2022@s2", "vs_E1-01_%"]
    rows22.append({"후보": f"{n} {desc}", "시드 0·1": round(a, 2), "시드 2·3": round(c, 2), "평균": round((a + c) / 2, 2)})
t22 = pd.DataFrame(rows22).set_index("후보")
print(f"2022-23 기준 모델 벌금: 시드 0·1 {bt.loc['E1-01@y2022', 'penalty']:,.0f}원, 시드 2·3 {bt.loc['E1-01@y2022@s2', 'penalty']:,.0f}원")
print(f"채점 발전소 {106}곳, 학습 행 287,999 (다른 해의 1/3 수준)")
t22

2022-23 기준 모델 벌금: 시드 0·1 57,408,602원, 시드 2·3 57,207,365원
채점 발전소 106곳, 학습 행 287,999 (다른 해의 1/3 수준)


,시드 0·1,시드 2·3,평균
후보,,,
E2-04 하루 요약,-2.12,-1.73,-1.93
E2-16 두 제공사,-2.90,-1.74,-2.32
E2-24 + 전날 정보,-2.84,-2.53,-2.68
E2-37 E2-24 − 전날 일교차,-2.86,-1.83,-2.34
E2-40 + 아침 안개 (최종 후보),-3.20,-3.46,-3.33
E2-46 E2-40 − 전날 일교차,-3.14,-2.69,-2.91


**규칙대로 판정**
1. E2-40(평균 −3.33%)은 E2-24(−2.68%)보다 나쁘지 않고 0.65%p 좋다 → **E2-40 유지.**
2. E2-24(−2.68%)는 E2-16(−2.32%)보다 나쁘지 않다 → 전날 정보도 이 해에서 유지.
3. 단순화 E2-46(−2.91%)은 E2-40보다 0.42%p 덜 좋다(시드 2·3에서 차이가 크다) → **채택하지 않는다.**
4. 모든 단계가 기준 모델보다 1.7% 이상 좋다.

**그리고 E2-40은 이 해의 시드 0·1과 2·3에서 모두 1위다.** 고르는 데 전혀 쓰지 않은 해에서도 1위이므로, 같은 해들로 반복해 고르면서 수치만 부풀린 결과는 아니라고 판단한다.

**솔직히 짚을 점**: 이 해의 개선 폭(−3.3%)은 다른 두 해(−6.4 ~ −7.0%)의 절반 정도다. 이 해는 학습 데이터가 16개월·발전소 106곳뿐이라 모델 자체가 약하다는 이유가 있다. 하지만 같은 해들로 골라서 생긴 낙관도 일부 섞여 있을 수 있다. 그래서 실제 평가 기간의 개선은 3 ~ 7% 사이로 본다.

## 민감도 점검 — 아침 안개 시간대(06~10시)는 우연히 맞은 것인가

06~10시는 11라운드에서 처음 정한 값이고, 다른 시간대와 비교해 고른 것이 아니다. 이웃한 창으로도 효과가 나는지 확인했다. **결과와 관계없이 창은 바꾸지 않는다고 미리 정했다.** 결과를 보고 가장 좋은 창으로 바꾸면 그것이 바로 수치 맞추기이기 때문이다.

In [22]:
rowsw = []
for n, desc in [("E2-24", "안개 신호 없음"), ("E2-47", "05~09시"), ("E2-40", "06~10시 (채택)"), ("E2-48", "07~11시")]:
    a = pct(score(f"{n}x2")["penalty"], score("E1-01x2")["penalty"])
    y0, y2 = bt.loc[f"{n}@y2022", "vs_E1-01_%"], bt.loc[f"{n}@y2022@s2", "vs_E1-01_%"]
    rowsw.append({"안개 창": f"{n} {desc}", "가상 평가연도 시드 0·1": round(a, 2),
                  "2022-23 시드 0·1": round(y0, 2), "2022-23 시드 2·3": round(y2, 2), "2022-23 평균": round((y0 + y2) / 2, 2)})
pd.DataFrame(rowsw).set_index("안개 창")

,가상 평가연도 시드 0·1,2022-23 시드 0·1,2022-23 시드 2·3,2022-23 평균
안개 창,,,,
E2-24 안개 신호 없음,-6.74,-2.84,-2.53,-2.68
E2-47 05~09시,-7.06,-2.85,-2.82,-2.84
E2-40 06~10시 (채택),-6.98,-3.20,-3.46,-3.33
E2-48 07~11시,-6.00,-3.09,-2.80,-2.94


**판단**
- 이른 아침 창(05~09시, 06~10시)은 두 해 모두에서 안개 신호가 없는 E2-24보다 좋다. 06~10시만 운 좋게 맞은 것이 아니다.
- 늦은 창(07~11시)은 가상 평가연도에서 오히려 E2-24보다 나쁘다. 안개·박무는 보통 오전 중반이면 걷히므로, 창이 늦을수록 신호가 흐려진다는 물리적 설명과 맞는다.
- 창은 미리 정한 대로 06~10시를 유지한다.

## 최종안 E2-40 — 월별로 고르게 좋아졌나

E2-24(중간 결론)와 E2-40(최종)의 월별 변화를 함께 본다. 모두 같은 시드 기준 모델 대비다.

In [23]:
c0, c2 = score("E1-01x2"), score("E1-01s2x2")
m = pd.DataFrame({"기준 벌금(시드 0·1)": c0["by_month"].round(0)})
for n in ["E2-24", "E2-40"]:
    m[f"{n}(시드 0·1)"] = pct(score(f"{n}x2")["by_month"], c0["by_month"]).round(2)
    m[f"{n}(시드 2·3)"] = pct(score(f"{n}s2x2")["by_month"], c2["by_month"]).round(2)
for n in ["E2-24", "E2-40"]:
    both = int(((m[f"{n}(시드 0·1)"] < 0) & (m[f"{n}(시드 2·3)"] < 0)).sum())
    print(f"{n}: 두 시드 모두 나아진 달 {both} / 12")
m

E2-24: 두 시드 모두 나아진 달 11 / 12
E2-40: 두 시드 모두 나아진 달 6 / 12


,기준 벌금(시드 0·1),E2-24(시드 0·1),E2-24(시드 2·3),E2-40(시드 0·1),E2-40(시드 2·3)
month,,,,,
2024-06,16872468.0,-10.77,-12.34,-15.96,-13.31
2024-07,25264787.0,-17.76,-16.42,-17.76,-16.27
2024-08,16745021.0,-0.42,-1.57,1.21,-1.97
2024-09,26258660.0,-9.48,-9.30,-9.57,-9.18
2024-10,19660277.0,-6.49,-4.89,-5.79,-6.31
2024-11,13595002.0,-2.24,-0.16,-1.76,1.31
2024-12,12695849.0,-1.02,-1.08,-0.91,-3.07
2025-01,13743491.0,1.10,1.70,0.91,3.99
2025-02,16372969.0,-2.95,-0.60,-0.43,0.41


**판단**
- E2-40은 5월(−16.5%, −17.6%)과 6월(−16.0%, −13.3%)에서 E2-24보다 크게 더 줄었다. 늦봄·초여름은 아침 안개·박무가 잦은 시기라 안개 신호의 효과와 맞는다. 7월(−17.8%, −16.3%)과 9월(−9.6%, −9.2%)의 큰 개선은 E2-24와 같은 수준으로 유지된다.
- 대신 개선이 덜 고르다. 두 시드에서 모두 나아진 달이 E2-24는 11개월인데 E2-40은 6개월이다. 8·11·2·3·4월은 시드에 따라 방향이 바뀌고(−3 ~ +1.3%), 1월은 두 시드 모두 나빠졌다(+0.9%, +4.0%).
- 연간 합계와 다른 해 검증(측정 4개 모두 E2-24보다 좋음)을 우선해 E2-40을 최종으로 삼는다. 월별로 덜 고르다는 점은 한계로 남긴다(맨 아래).

## 최종 피처 — 무엇을 계산하나

In [24]:
for f in (fw.add_day_summary, fw.add_day_summary_2prov_nocloud, fw.add_e16_prev_rain, fw._lag_day,
          fw.add_e21_prev_more, fw.add_e24_fog, fw.add_weather_features):
    print(inspect.getsource(f))

def add_day_summary(df):                                       # E2-04 하루 요약
    """발전소 × 대상일 요약: 해 뜬 시간 평균 운량, 일교차, 하루 강수 합. 전부 같은 대상일 안의 예보."""
    key = [df.site_id, df.date]
    df["wx_day_cloud"] = df.wa_cloud_r17.where(df.elev > 0).groupby(key).transform("mean")
    t = df.wa_temp_r17.groupby(key)
    df["wx_day_trange"] = t.transform("max") - t.transform("min")
    df["wx_day_rain"] = df.wb_total_liq_r17.groupby(key).transform("sum")
    return df

def add_day_summary_2prov_nocloud(df):                         # E2-16 E2-15 에서 B사 하루 운량 빼기
    df = add_day_summary(df)
    key = [df.site_id, df.date]
    t = df.wb_temp_r17.groupby(key)
    df["wx_day_trange_b"] = t.transform("max") - t.transform("min")
    df["wx_day_rain_a"] = df.wa_rain_r17.groupby(key).transform("sum")
    return df

def add_e16_prev_rain(df):                                     # E2-21 + 전날 강수 합 (전 대상일 예보 → 사용 가능)
    df = add_day_summary_2prov_nocloud(df)
    d = df.groupby(["site_id", "date"], observed=True)

| 피처 | 뜻 | 계산 |
|---|---|---|
| `wx_day_cloud` | 그날 해 뜬 시간의 평균 운량 (A사) | `wa_cloud_r17`, elev > 0 인 시각의 발전소 × 대상일 평균 |
| `wx_day_trange` | 그날 일교차 (A사) | `wa_temp_r17` 의 하루 최고 − 최저 |
| `wx_day_trange_b` | 그날 일교차 (B사) | `wb_temp_r17` 의 하루 최고 − 최저 |
| `wx_day_rain` | 그날 강수 합 (B사) | `wb_total_liq_r17` 의 하루 합 |
| `wx_day_rain_a` | 그날 강수 합 (A사) | `wa_rain_r17` 의 하루 합 |
| `wx_prev_rain` | 전날 강수 합 (B사) | 전 대상일의 `wx_day_rain` |
| `wx_prev_rain_a` | 전날 강수 합 (A사) | 전 대상일의 `wx_day_rain_a` |
| `wx_prev_cloud` | 전날 평균 운량 (A사) | 전 대상일의 `wx_day_cloud` |
| `wx_prev_trange` | 전날 일교차 (A사) | 전 대상일의 `wx_day_trange` |
| `wx_am_vis_min` | 그날 아침 최저 시정 (B사) | `wb_vis_r17`, 06~10시(ts 기준) 최솟값 |
| `wx_am_humid` | 그날 아침 평균 습도 (A사) | `wa_humidity_r17`, 06~10시(ts 기준) 평균 |

모두 17시 발행분 예보만 쓴다. 그날 값과 아침 값은 같은 대상일(`date`, 01:00 ~ 다음날 00:00) 안에서, 전날 값은 바로 앞 대상일에서 계산한다. 전 대상일 예보는 D−2일 17시에 발행되어, D일 예측시점(D−1일 17시)보다 먼저 나와 있다.

## 누설 점검 — 쓰면 안 되는 날의 예보가 섞이지 않았나

한 발전소의 10일치를 꺼내 최종 함수를 적용한 뒤, 날짜 구간별로 예보를 일부러 망가뜨리고 다시 적용한다.
1. 기준일 **이후** 날짜를 망가뜨림 → 기준일까지의 피처가 그대로여야 한다(미래 대상일 예보 미사용).
2. 기준일 **이틀 전 이전**을 망가뜨림 → 기준일 피처가 그대로여야 한다(그날·전날만 사용).
3. 기준일 **전날**을 망가뜨림 → 기준일의 "그날" 피처는 그대로이고 "전날" 피처만 바뀌어야 한다.

In [25]:
cols = ["site_id", "ts", "date", "elev", "hour", "wa_cloud_r17", "wa_temp_r17", "wb_temp_r17",
        "wb_total_liq_r17", "wa_rain_r17", "wb_vis_r17", "wa_humidity_r17"]
base = pd.read_parquet(ROOT / "outputs" / "work" / "base.parquet", columns=cols)
t0 = pd.Timestamp("2024-07-01 01:00", tz="Asia/Seoul")
sub = base[(base.site_id == "PV001") & (base.ts >= t0) & (base.ts < t0 + pd.Timedelta(days=10))].reset_index(drop=True)
sub["site_id"] = sub.site_id.astype(str)
del base

days = sorted(sub.date.unique())
D, Dm1 = days[5], days[4]
fc = ["wa_cloud_r17", "wa_temp_r17", "wb_temp_r17", "wb_total_liq_r17", "wa_rain_r17", "wb_vis_r17", "wa_humidity_r17"]
today = ["wx_day_cloud", "wx_day_trange", "wx_day_trange_b", "wx_day_rain", "wx_day_rain_a", "wx_am_vis_min", "wx_am_humid"]
prev = ["wx_prev_rain", "wx_prev_rain_a", "wx_prev_cloud", "wx_prev_trange"]
ref = fw.add_weather_features(sub.copy())

def broken(mask):
    x = sub.copy()
    x.loc[mask, fc] = x.loc[mask, fc] * 3 + 17
    return fw.add_weather_features(x)

same = lambda p, m, c: np.allclose(ref.loc[m, c], p.loc[m, c], equal_nan=True)
fut, old, yday = broken(sub.date > D), broken(sub.date < Dm1), broken(sub.date == Dm1)
onD = sub.date == D
print(f"점검 대상: PV001, {days[0]:%Y-%m-%d} ~ {days[-1]:%Y-%m-%d} ({len(sub)}행), 기준일 {D:%Y-%m-%d}")
print("1. 이후 날짜를 바꿔도 기준일까지 피처 그대로:      ", same(fut, sub.date <= D, today + prev))
print("2. 이틀 전 이전을 바꿔도 기준일 피처 그대로:       ", same(old, onD, today + prev))
print("3. 전날을 바꾸면 기준일 '그날' 피처는 그대로:      ", same(yday, onD, today),
      " / '전날' 피처는 바뀜:", not same(yday, onD, prev))
print("행 수·순서 유지:", len(ref) == len(sub) and (ref.ts.values == sub.ts.values).all())
print("실측 기상(observed_*) 사용:", any("observed" in c for c in ref.columns))

점검 대상: PV001, 2024-07-01 ~ 2024-07-10 (240행), 기준일 2024-07-06
1. 이후 날짜를 바꿔도 기준일까지 피처 그대로:       True
2. 이틀 전 이전을 바꿔도 기준일 피처 그대로:        True
3. 전날을 바꾸면 기준일 '그날' 피처는 그대로:       True  / '전날' 피처는 바뀜: True
행 수·순서 유지: True
실측 기상(observed_*) 사용: False


## 재현과 실제 평가 기간 실행 확인

- `E2-finalx2`: 전달용 함수 `add_weather_features` 하나로 다시 돌린 결과. E2-40과 같아야 한다.
- `outputs/preds_test/E2-finalx2.parquet`: 같은 설정으로 2025-05까지 학습해 실제 평가 기간(253곳 × 8,760시각)을 예측한 결과. 행 수와 결측을 확인한다.

In [26]:
fin, e40 = score("E2-finalx2"), score("E2-40x2")
pa = read_preds(PREDS / "E2-finalx2.parquet").sort_values(["site_id", "ts"]).pred_kwh.to_numpy()
pb = read_preds(PREDS / "E2-40x2.parquet").sort_values(["site_id", "ts"]).pred_kwh.to_numpy()
print(f"E2-finalx2 벌금 {fin['penalty']:,.0f}원 / E2-40x2 {e40['penalty']:,.0f}원, 예측 최대 차이 {np.abs(pa - pb).max():.6f} kWh")

t = pd.read_parquet(ROOT / "outputs" / "preds_test" / "E2-finalx2.parquet")
print(f"평가 기간 예측 {len(t):,}행 (기대 {253 * 8760:,}), 발전소 {t.site_id.nunique()}곳, "
      f"결측 {int(t.pred_kwh.isna().sum())}, 음수 {int((t.pred_kwh < 0).sum())}, 1년 합계 {t.pred_kwh.sum() / 1e6:.2f} GWh")

E2-finalx2 벌금 199,117,580원 / E2-40x2 199,117,580원, 예측 최대 차이 0.000000 kWh


평가 기간 예측 2,216,280행 (기대 2,216,280), 발전소 253곳, 결측 0, 음수 0, 1년 합계 59.92 GWh


## 전달본 — 읽기 쉬운 단독 파일

`src_tools/features_weather.py`에는 실험용 함수가 50개 가까이 있다. 노트북과 `outputs/preds/E2-*.json`이 참조하므로 지우지 않는다. 전달 대상인 `add_weather_features`만 담은 단독 정리본을 `analysis_lab/E2_handover/features_weather.py`에 따로 두었다. 두 파일이 같은 피처를 내는지 검증 구간 전체로 대조한다.

In [27]:
import importlib.util
spec = importlib.util.spec_from_file_location("handover", ROOT / "analysis_lab" / "E2_handover" / "features_weather.py")
ho = importlib.util.module_from_spec(spec); spec.loader.exec_module(ho)
fr = tsm.load_frame("valid", 2)
x1 = fw.add_weather_features(fr.copy())
x2 = ho.add_weather_features(fr.copy())
c1 = [c for c in x1.columns if c.startswith("wx_")]
c2 = [c for c in x2.columns if c.startswith("wx_")]
print("피처 열 이름·순서 같음:", c1 == c2)
print("값 완전히 같음:", all(np.array_equal(x1[c].to_numpy(), x2[c].to_numpy(), equal_nan=True) for c in c1))
print(f"대조한 행 {len(x1):,}, 피처 {c1}")
del fr, x1, x2

피처 열 이름·순서 같음: True


값 완전히 같음: True
대조한 행 4,104,567, 피처 ['wx_day_cloud', 'wx_day_trange', 'wx_day_rain', 'wx_day_trange_b', 'wx_day_rain_a', 'wx_prev_rain', 'wx_prev_rain_a', 'wx_prev_cloud', 'wx_prev_trange', 'wx_am_vis_min', 'wx_am_humid']


## 실험을 다시 돌리는 명령

이 노트북은 `outputs/preds/` 에 저장된 예측을 채점만 한다(8라운드 진단 셀을 빼면 학습이 없다). 예측 파일(parquet)은 git에 올라가지 않으므로, 다른 PC에서는 아래처럼 먼저 만든다. 저장소 맨 위 폴더에서 실행하며, 실험 하나는 이 PC에서 약 2~3분이다.

```
# 피처 실험 (기준 모델 + 함수 하나, 시드 0·1)
python src_tools/train_site_model.py --exp E2-40x2 --base champion --seeds 2 --add features_weather:add_e24_fog
# 같은 설정을 시드 2·3으로
python src_tools/train_site_model.py --exp E2-40s2x2 --base E2-40x2 --seed 2 --seeds 2
# 전달용 함수로 재현, 실제 평가 기간 예측
python src_tools/train_site_model.py --exp E2-finalx2 --base champion --seeds 2 --add features_weather:add_weather_features
python src_tools/train_site_model.py --exp E2-finalx2 --base champion --seeds 2 --add features_weather:add_weather_features --mode test
# 다른 해 백테스트 (결과는 analysis_lab/E2_backtest_results.csv). 맨 앞은 항상 E1-01
python analysis_lab/E2_backtest.py E1-01 E2-04 E2-16 E2-21 E2-24 E2-27 E2-28 E2-30 E2-37 E2-40
# 백테스트를 시드 2·3으로 (PowerShell: $env:BT_SEED0="2"; 결과 이름 끝에 @s2)
BT_SEED0=2 python analysis_lab/E2_backtest.py E1-01 E2-24 E2-30 E2-37 E2-40
```

실험별 함수 이름은 `src_tools/features_weather.py` 의 주석(`# E2-xx`)에 적혀 있다. 쌓기 실험(E2-12·13)은 `--base E2-04x2` 위에 `--add` 를 하나 더 붙였다.

## 전체 실험 한눈에 (기준 모델 대비, 시드 0·1)

In [28]:
ALL = R1 + R2[1:] + [("E2-11x2", "운량 + 일교차 (강수 합 뺌)"),
                     ("E2-12x2", "E2-04 + 전 발전소 평균 운량"),
                     ("E2-13x2", "E2-04 + 두 발행분 평균·차이"),
                     ("E2-14x2", "하루 요약을 두 발행분 평균값으로"),
                     ("E2-15x2", "E2-04 + B 하루 운량 + B 일교차 + A 하루 강수 합"),
                     ("E2-16x2", "E2-15에서 B 하루 운량 뺌 (중간 결론)"),
                     ("E2-17x2", "E2-15 + 이슬점 차이"),
                     ("E2-18x2", "E2-16 + 운량 표준편차"),
                     ("E2-19x2", "E2-16 + 비 온 시간 수"),
                     ("E2-20x2", "E2-16, 일교차를 해 뜬 시간으로"),
                     ("E2-21x2", "E2-16 + 전날 강수 합"),
                     ("E2-22x2", "E2-16 + 전 발전소 하루 평균"),
                     ("E2-23x2", "E2-16 + 강수확률"),
                     ("E2-24x2", "E2-21 + 전날 운량·일교차 (중간 결론)"),
                     ("E2-25x2", "E2-21 + 이틀 전 강수 합"),
                     ("E2-26x2", "E2-21, 일교차를 해 뜬 시간으로"),
                     ("E2-27x2", "E2-24를 관측지점 기준으로"),
                     ("E2-28x2", "E2-27 + 전날 B 일교차"),
                     ("E2-29x2", "E2-24 + 오늘−전날 변화량"),
                     ("E2-30x2", "E2-24 + 습도 (오늘·전날)"),
                     ("E2-31x2", "E2-24 + 기압과 변화"),
                     ("E2-32x2", "E2-24 + 풍속 (오늘·전날)"),
                     ("E2-33x2", "E2-24, 강수를 해 뜬 시간만으로"),
                     ("E2-34x2", "E2-24 − A사 강수"),
                     ("E2-35x2", "E2-24 − B사 일교차"),
                     ("E2-36x2", "E2-24 − 전날 운량"),
                     ("E2-37x2", "E2-24 − 전날 일교차"),
                     ("E2-38x2", "E2-24 + 강설"),
                     ("E2-39x2", "E2-24 + 밤사이 강수"),
                     ("E2-40x2", "E2-24 + 아침 안개 신호 ← 최종"),
                     ("E2-41x2", "E2-24 + 주변 발전소 하루 요약"),
                     ("E2-42x2", "E2-24, 시각별 풍속·기압 빼기"),
                     ("E2-43x2", "E2-24 + 아침 최저 시정만"),
                     ("E2-44x2", "E2-24 + 아침 습도만"),
                     ("E2-45x2", "E2-40 + 아침 최저 운고"),
                     ("E2-46x2", "E2-40 − 전날 일교차 (단순화)"),
                     ("E2-47x2", "안개 창 05~09시 (민감도 점검)"),
                     ("E2-48x2", "안개 창 07~11시 (민감도 점검)")]
allt = table(ALL)
allt["_p"] = allt["벌금(원)"].str.replace(",", "").astype(float)
allt.sort_values("_p").drop(columns="_p")

,바꾼 것,벌금(원),E1-01x2 대비,11~15시 대비,나아진 달,판정
실험,,,,,,
E2-47x2,안개 창 05~09시 (민감도 점검),"198,942,978",-7.06%,-7.09%,9/12,채택 기준 충족
E2-40x2,E2-24 + 아침 안개 신호 ← 최종,"199,117,580",-6.98%,-7.09%,9/12,채택 기준 충족
E2-24x2,E2-21 + 전날 운량·일교차 (중간 결론),"199,632,154",-6.74%,-6.93%,11/12,채택 기준 충족
E2-45x2,E2-40 + 아침 최저 운고,"199,836,444",-6.65%,-6.52%,10/12,채택 기준 충족
E2-46x2,E2-40 − 전날 일교차 (단순화),"199,978,072",-6.58%,-6.66%,11/12,채택 기준 충족
E2-30x2,E2-24 + 습도 (오늘·전날),"200,415,887",-6.38%,-6.58%,10/12,채택 기준 충족
E2-41x2,E2-24 + 주변 발전소 하루 요약,"200,448,157",-6.36%,-6.33%,10/12,채택 기준 충족
E2-28x2,E2-27 + 전날 B 일교차,"200,456,473",-6.36%,-6.47%,12/12,채택 기준 충족
E2-39x2,E2-24 + 밤사이 강수,"200,634,996",-6.27%,-6.33%,9/12,채택 기준 충족


In [29]:
print("기록 시트에 붙여넣기 (실험, 벌금, 기준 대비, 11~15시 대비) — 시드 2·3 결과는 같은 시드 기준 모델(E1-01s2x2) 대비")
b = score("E1-01x2")
SEED2 = ["E2-04s2x2", "E2-11s2x2", "E2-15s2x2", "E2-16s2x2", "E2-21s2x2", "E2-24s2x2", "E2-27s2x2", "E2-28s2x2",
         "E2-30s2x2", "E2-37s2x2", "E2-40s2x2", "E2-46s2x2"]
for exp, _ in ALL + [(x, "") for x in SEED2] + [("E2-finalx2", "")]:
    r = score(exp)
    ref = score("E1-01s2x2") if "s2" in exp else b
    print(f"{exp}\t{r['penalty']:.0f}\t{pct(r['penalty'], ref['penalty']):+.2f}%\t{pct(r['peak'], ref['peak']):+.2f}%")

기록 시트에 붙여넣기 (실험, 벌금, 기준 대비, 11~15시 대비) — 시드 2·3 결과는 같은 시드 기준 모델(E1-01s2x2) 대비
E2-02x2	214141791	+0.04%	+0.24%
E2-03x2	213885743	-0.08%	-0.05%
E2-04x2	209676101	-2.05%	-1.78%
E2-05x2	213992844	-0.03%	+0.16%
E2-05bx2	214972282	+0.42%	+0.65%
E2-06x2	213591756	-0.22%	-0.14%
E2-07x2	214408373	+0.16%	+0.11%
E2-04ax2	212964069	-0.51%	-0.47%
E2-04bx2	212595429	-0.69%	-0.43%
E2-04cx2	214472129	+0.19%	+0.27%
E2-08x2	211157656	-1.36%	-1.10%
E2-09x2	209944857	-1.92%	-1.41%
E2-10x2	212344840	-0.80%	-0.31%
E2-11x2	212255290	-0.84%	-0.57%
E2-12x2	210214015	-1.80%	-1.56%
E2-13x2	209740385	-2.02%	-1.82%
E2-14x2	214412243	+0.16%	+0.67%
E2-15x2	207428310	-3.10%	-3.03%
E2-16x2	206704167	-3.44%	-3.24%
E2-17x2	208009515	-2.83%	-2.56%
E2-18x2	206194643	-3.68%	-3.63%
E2-19x2	207100773	-3.25%	-3.19%
E2-20x2	206069846	-3.73%	-3.65%
E2-21x2	202444653	-5.43%	-5.38%
E2-22x2	209531507	-2.12%	-1.66%
E2-23x2	206634358	-3.47%	-3.40%
E2-24x2	199632154	-6.74%	-6.93%
E2-25x2	201674928	-5.79%	-5.88%
E2-26x2	202657331	-5.3

## 최종 정리

**채택: `features_weather.add_weather_features` (= E2-40, 하루 요약 피처 11개)**

| 기준 | 결과 |
|---|---|
| 가상 평가연도, 시드 0·1 | 199,117,580원, 기준 모델 대비 −6.98%, 11~15시 −7.09% |
| 가상 평가연도, 시드 2·3 | 200,220,943원, 같은 시드 기준 모델 대비 −6.65%, 11~15시 −6.56% |
| 가상 평가연도, 시드 4개 평균 | −6.83%, 11~15시 −6.83%, 12개월 중 11개월 개선 |
| 다른 해(2023-06 ~ 2024-05) 백테스트 | 시드 0·1 −6.41%, 시드 2·3 −6.38% |
| 측정 4개 평균 | −6.61% (직전 최종안 E2-24 −6.19%, 네 측정 모두 E2-40이 좋음) |
| 마지막 시험 2022-23 (선택에 미사용) | 시드 0·1 −3.20%, 시드 2·3 −3.46%, 비교 후보 6개 중 두 시드 모두 1위 |
| 안개 시간대 민감도 | 05~09시도 E2-24보다 좋음, 07~11시는 약함 (물리적 설명과 일치) |
| 팀 채택 규칙 | 충족 (−0.5% 이하, 11~15시도 감소) |
| 누설 점검 | 미래 대상일·이틀 전 이전 예보를 바꿔도 피처 불변, 실측 기상 미사용 |
| 재현·실행 | 전달용 함수로 같은 결과 재현, 실제 평가 기간 2,216,280행 예측 정상, 단독 정리본과 값 완전히 일치 |

**왜 이것인가**
1. 시험한 49개 조합 중 측정 4개 평균과 가상 평가연도 시드 4개 평균 모두 1위다. 처음 기준을 넘은 E2-04(−1.79%, 시드 4개)보다 개선 폭이 거의 네 배다.
2. **고르는 데 쓰지 않은 2022-23년에서도 1위다.** 결과를 보기 전에 정한 규칙으로 확인했다. 같은 해들로 반복해 고르면서 수치만 부풀린 결과가 아니라는 가장 강한 근거다.
3. 두 해·두 시드 쌍 네 측정이 −6.4 ~ −7.0%로 고르다(폭 0.60%p). 직전 최종안 E2-24는 −5.4 ~ −6.7%였다.
4. 꼭짓점이다. 안개 신호를 쪼개도(12라운드), 다른 정보를 더해도(9·11·12라운드), E2-24의 묶음을 빼도(10라운드), 전날 일교차를 빼도(E2-46) 나빠지거나 마지막 시험에서 덜 좋았다.
5. 이유를 설명할 수 있다.
   - 한 시각의 예보보다 "그날이 맑고 건조한 날인지"를 알려주는 하루 단위 값이 발전량을 더 잘 가른다.
   - 비를 몰고 온 기상 시스템은 하루 만에 사라지지 않는다. 그래서 전날 예보의 강수·운량·일교차가 오늘 예보를 얼마나 믿을지에 대한 신호가 된다.
   - 아침의 낮은 시정과 높은 습도가 함께 나타나면 안개·박무다. 오전 발전을 깎고, 늦봄·초여름(5·6월)에 효과가 가장 컸다.

**채택하지 않은 것과 이유**
- 시각 단위 피처(운고, 앞뒤 시각, 라운드 요약, 전 지점 평균, 제공사 차이, 자외선지수 앞뒤): 기준 모델 대비 ±0.4% 안이거나 나빠짐
- 그날 하루 변수를 더 얹기(B 운량·습도·강수확률·자외선, 오전·오후, 하루 평균 대비 차이, 이슬점 차이, 운량 표준편차, 비 온 시간 수, 전 발전소 평균): 잡음 범위이거나 나빠짐
- 강수 합 빼기, 발행분 평균으로 계산하기, 해 뜬 시간 일교차, 이틀 전 강수: 개선이 줄거나 없음
- 관측지점 기준 계산(E2-27·28): 착시 점검용으로 만들었다. 성능은 비슷하지만 측정마다 흔들림이 더 커서 최종으로 고르지 않았다.
- 변화량·습도·기압·풍속 더하기(9라운드): 모두 나빠짐
- E2-24에서 묶음 빼기(10라운드): 모두 나빠짐
- 강설, 밤사이 강수, 주변 발전소 하루 요약 더하기(11라운드): 나빠짐. 시각별 풍속·기압 빼기도 나빠짐
- 안개 신호를 시정만·습도만 쓰기, 아침 운고 더하기(12라운드): 나빠짐

**한계**
- 개선 폭이 해마다 다르다. 2023-24·2024-25년은 −6.4 ~ −7.0%, 마지막 시험 2022-23년은 −3.3%였다. 2022-23년은 학습 데이터가 짧아서이기도 하지만, 같은 해들로 골라서 생긴 낙관도 섞여 있을 수 있다. **실제 평가 기간의 개선은 3 ~ 7% 사이로 예상한다.**
- E2-40은 월별로 덜 고르다. 두 시드에서 모두 나아진 달이 6개월이고(E2-24는 11개월), 1월은 두 시드 모두 나빠졌다. 연간 개선의 상당 부분이 5·6·7·9월에서 나온다.
- 다른 해에는 발전소가 205곳뿐이다.
- 하루 요약·전날 값은 학습 행(실적이 있는 시각)으로 계산하고, 실제 평가 기간은 전 시각 격자로 계산한다. 8라운드에서 차이가 1% 미만의 행에서만 생기는 것을 확인했다.
- 기준 모델(`champion`)이 바뀌면 이 효과도 다시 확인해야 한다.

**탐색을 여기서 멈춘 이유**
- 49개 조합을 2024-25년과 2023-24년으로 반복해 비교하며 골랐다. 이 두 해는 이미 판단에 쓴 데이터라, 여기서 더 고르면 그 해들에 맞추는 과적합이 된다.
- 마지막 시험에 쓴 2022-23년도 이제 판단에 썼다. 남은 2021-22년은 데이터가 1년이 안 되고 발전소가 31 ~ 80곳뿐이라, 새 시험연도로 쓸 수 없다.
- 그래서 새로운 확인용 데이터 없이 추가로 피처를 고르는 것은 수치만 끌어올릴 위험이 크다고 판단해 멈췄다. 최근 라운드의 개선 폭(0.3 ~ 0.4%p)도 시드 잡음(±0.5%)과 비슷해졌다.
- 앞으로 다시 실험한다면, 기준 모델이 바뀌었을 때(예: ④의 모델 교체) 이 피처의 효과가 유지되는지 확인하는 정도가 적절하다.

**전달 (11/6)**
- ①에게 `src_tools/features_weather.py`를 넘긴다. 실행은 `--add features_weather:add_weather_features`다. 읽기용 단독 정리본은 `analysis_lab/E2_handover/features_weather.py`다(같은 결과).
- 다른 해 백테스트 도구 `analysis_lab/E2_backtest.py`도 함께 넘긴다. 다른 역할의 피처도 같은 방식으로 다른 해에서 확인할 수 있다.
- 위 "기록 시트에 붙여넣기" 줄을 실험 기록 시트에 올린다.
- 팀 채팅에서 채택 후보를 "E2-40(`add_weather_features`), 측정 4개 평균 −6.6%"로 알린다.